Describe the environment in the Nim learning model

The states are 3 piles that can each have from 0 to 10 items on it each
The actions are removing x items from a pile
The state where the game ends is then all piles have 0 items

Describe the agent(s) in the Nim learning model (Hint, not just the Q-learner). Is Guru an agent?

All three players are agents, the random player guru and q learner.

The random player is a stochastic agent which simply randomly chooses which pile and how many items to move. It has a fixed ranom policy.

The q learner is a RL agent that tries learning the optimal policy through eploring and exploiting over games. It uses the Q-table to pick the best move and explores using a random move if no good move is found.

The guru is a rule based agent using a specific optimal strategy. The strategy it uses is the nim solution, where it computes the XOR of all piles.

Describe the reward and penalty in the Nim learning model.

When the Q learner wins the game by clearing the last pile, it will receive a reward of 100. If a state is not one that is a game ender, there will be no reward so it gets a reward of 0. If the q learner doesn't make the optimal move and results in a loss, it doesn't get the reward of 100, which acts as a penalty.

How many possible states there could be in the Nim game with a maximum of 10 items per pile and 3 piles total?

There are 3 piles with a max of 10 items each, therefore for each pile there are 11 states.
The total states is 11^3 = 1331

How many possible unique actions are there for player 1 to take as their first action in a Nim game with 10 items per pile and 3 piles total?

Each pile has 10 items and for a move at least one item must be removed from a pile. Therefore player 1 can remove anwhere from 1-10 items from any one of the piles. This means that there are 10 possibile actions for each of the 3 piles.
Therefore
10*3 = 30
30 possible unique actions

Do you think a Q-learner can beat the Guru player? Why or why not?

I don't think Q-learner can beat the Guru player
The game is deterministic, meaning that the same state and action will result in the same outcome, there is no randomness factor to the game. Guru plays in the most optimal way in a game that is deterministic which makes it tough for the Q-learner player to beat the Guru player. The Q-learner table is also massive and it takes time to fill the table the table for all states and actions, the Q-learner only see optimal moves from the previous states its been in.
Q-learner may beat the Guru player in some instances, but to consistently beat the Guru player is unlikely.

Find a way to improve the provided Nim game learning model. (Hint: How about penalizing the losses? Hint: It is indeed possible to find a better solution, which improves the way Q-learning updates its Q-table)

I tried penalizing losses by giving a loss penalty of -60.

I also tried keeping a history of the moves in a game and using this history to update the q table at the end with the reards and penalties. This is to improve the way q learning updates its q table


The following is my code.

In [1]:
import numpy as np
from random import randint, choice

ITEMS_MX = 10

Alpha, Gamma, Reward, Loss_Penalty = 1.0, 0.8, 100.0, -60.0
qtable = None


def init_game() -> list:
    return [randint(1, ITEMS_MX), randint(1, ITEMS_MX), randint(1, ITEMS_MX)]


def nim_guru(_st: list) -> (int, int):
    xored = _st[0] ^ _st[1] ^ _st[2]
    if xored == 0:
        return nim_random(_st)
    for pile in range(3):
        s = _st[pile] ^ xored
        if s <= _st[pile]:
            return _st[pile] - s, pile


def nim_random(_st: list) -> (int, int):
    pile = choice([i for i in range(3) if _st[i] > 0])
    return randint(1, _st[pile]), pile


def nim_qlearner(_st: list) -> (int, int):
    global qtable
    a = np.argmax(qtable[_st[0], _st[1], _st[2]])
    move, pile = a % ITEMS_MX + 1, a // ITEMS_MX

    if move <= 0 or _st[pile] < move:
        move, pile = nim_random(_st)

    return move, pile


Engines = {"Random": nim_random, "Guru": nim_guru, "Qlearner": nim_qlearner}


def game(_a: str, _b: str):
    state, side = init_game(), "A"
    move_history = []
    while True:
        engine = Engines[_a] if side == "A" else Engines[_b]
        move, pile = engine(state)
        move_history.append((tuple(state), move, pile))
        state[pile] -= move

        if state == [0, 0, 0]:
            end_qtable_update(state, move_history, side == "A")
            return side
        side = "B" if side == "A" else "A"


def play_games(_n: int, _a: str, _b: str) -> (int, int):
    from collections import defaultdict
    wins = defaultdict(int)
    for _ in range(_n):
        wins[game(_a, _b)] += 1
    print(f"{_n} games, {_a:>8s}{wins['A']:5d}  {_b:>8s}{wins['B']:5d}")
    return wins["A"], wins["B"]


def nim_qlearn(_n: int):
    global qtable
    qtable = np.zeros((ITEMS_MX + 1, ITEMS_MX + 1, ITEMS_MX + 1, ITEMS_MX * 3), dtype=np.float32)

    for _ in range(_n):
        st1 = init_game()
        move_history = []
        while True:
            move, pile = nim_random(st1)
            move_history.append((tuple(st1), move, pile))
            st2 = list(st1)
            st2[pile] -= move
            if st2 == [0, 0, 0]:
                end_qtable_update(st2, move_history, True)
                break

            qtable_update(0, st1, move, pile, np.max(qtable[st2[0], st2[1], st2[2]]))
            st1 = st2


def qtable_update(r: float, _st1: list, move: int, pile: int, q_future_best: float):
    a = pile * ITEMS_MX + move - 1
    qtable[_st1[0], _st1[1], _st1[2], a] = Alpha * (r + Gamma * q_future_best)


def end_qtable_update(state, move_history, won:bool):
    reward = Reward if won else Loss_Penalty
    for st1, move, pile in reversed(move_history):
        q_future_best = np.max(qtable[state[0],state[1],state[2]])
        qtable_update(reward,list(st1),move,pile,q_future_best)
        reward *=Gamma

nim_qlearn(10000) 

play_games(1000, "Qlearner", "Random")
play_games(1000, "Random", "Qlearner")
play_games(1000, "Qlearner", "Guru")
play_games(1000, "Guru", "Qlearner")


1000 games, Qlearner  635    Random  365
1000 games,   Random  506  Qlearner  494
1000 games, Qlearner    1      Guru  999
1000 games,     Guru  995  Qlearner    5


(995, 5)

In [2]:
%%time

# See the training size effect
n_train = (3, 10, 100, 1000, 10000, 50000, 100000)
Wins = []
for n in n_train:
    nim_qlearn(n)
    wins_a, wins_b = play_games(1000, 'Qlearner', 'Random')
    Wins += [wins_a/(wins_a+wins_b)]

1000 games, Qlearner  573    Random  427
1000 games, Qlearner  510    Random  490
1000 games, Qlearner  518    Random  482
1000 games, Qlearner  584    Random  416
1000 games, Qlearner  602    Random  398
1000 games, Qlearner  555    Random  445
1000 games, Qlearner  562    Random  438
CPU times: user 6.26 s, sys: 7.83 ms, total: 6.27 s
Wall time: 6.27 s


In [3]:
print(Wins)


[0.573, 0.51, 0.518, 0.584, 0.602, 0.555, 0.562]
